# خطة التدريب العلاجية المخصصة على Colab T4 للوكلاء المقصّرين
- يقرأ هذا الدفتر تلقائياً ملف `datasets/swarm10/colab_retrain_plan.json` الذي أنشأته ورفعته شاشة فحص السرب الـ 10.
- يقوم بتدريب الوكلاء الذين رُصد قصورهم في الدردشة أو غياب محولات LoRA الخاصة بهم فقط، ثم يدفع المحولات المحدثة مباشرة إلى GitHub.


In [ ]:
%cd /content
import os, json, subprocess
from pathlib import Path

REPO_URL = "https://github.com/sdxdxa56-design/jdjsjsh.git"
if not Path("/content/zh_swarm10/.git").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/zh_swarm10"], check=True)
else:
    subprocess.run(["git", "-C", "/content/zh_swarm10", "pull", "--ff-only"], check=False)
%cd /content/zh_swarm10
!pip install -q --upgrade transformers peft accelerate datasets safetensors bitsandbytes

plan_path = Path("datasets/swarm10/colab_retrain_plan.json")
if plan_path.exists():
    plan = json.loads(plan_path.read_text(encoding="utf-8"))
    print("📋 خطة التدريب المرفوعة من ZeroHunter:", json.dumps(plan, ensure_ascii=False, indent=2))
else:
    print("⚠️ لم يُعثر على colab_retrain_plan.json، سيتم تدريب جميع الوكلاء الـ 10.")


In [ ]:
# تنفيذ التدريب الموجه حسب خطة colab_retrain_plan.json وسجلات الدردشة المحفوظة
!python3 -m swarm10.colab_train_10_lora --use-plan --output-dir lora_adapters


In [ ]:
# التحقق من الأحجام (<50MB) ودفع المحولات المحدثة إلى GitHub
import getpass
from pathlib import Path
import subprocess

for p in sorted(Path("lora_adapters").rglob("*.safetensors")):
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f"Adapter: {p} -> {size_mb:.2f} MB")
    assert size_mb < 50, f"حجم المحول {p} تجاوز حد GitHub الآمن!"

gh_token = os.environ.get("GITHUB_TOKEN") or getpass.getpass("أدخل GitHub Token (صلاحية repo): ")
subprocess.run(["git", "config", "user.name", "ZeroHunter Swarm10 Trainer"], check=True)
subprocess.run(["git", "config", "user.email", "swarm10@zerohunter.internal"], check=True)
subprocess.run(["git", "add", "-f", "lora_adapters/", "datasets/swarm10/"], check=True)
subprocess.run(["git", "commit", "-m", "Apply targeted Colab LoRA retraining plan for weak agents"], check=False)
subprocess.run(["git", "push", f"https://{gh_token}@github.com/sdxdxa56-design/jdjsjsh.git", "HEAD:main"], check=True)
print("✅ تم تدريب الوكلاء المقصّرين ودفع المحولات المحدثة إلى GitHub بنجاح!")
